In [1]:
# Basic system utilities for interacting with files
# ----------------------General Imports------------------------------------
import glob, os, importlib
import time

# Numpy
import numpy as np

# To display full ouptut of cell, not just the last result
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# -----------------------Astropy/Astroquery Imports--------------------------------
# for viewing astropy images and tables
from astropy.io import fits
from astropy.table import Table


# for JWST calibration pipeline
import jwst
import crds

from jwst.pipeline import Detector1Pipeline
from jwst.pipeline import Image2Pipeline
from jwst.pipeline import Coron3Pipeline

# JWST pipeline utilities
from jwst import datamodels
from jwst.associations import asn_from_list  # Tools for creating association files
from jwst.associations.lib.rules_level3 import Asn_Lv3NRCCoron  # Definition of a Lvl3 association file

# Echo pipeline version and CRDS context in use
print(f"JWST Calibration Pipeline Version: {jwst.__version__}")
print(f"Using CRDS Context: {crds.get_context_name('jwst')}")




JWST Calibration Pipeline Version: 3.0.0


CRDS - INFO -  Calibration SW Found: jwst 3.0.0 (/home/cperrot/anaconda3/envs/jwst/lib/python3.13/site-packages/jwst-3.0.0.dist-info)


Using CRDS Context: jwst_1581.pmap


In [2]:
# --------------------------Set Processing Steps--------------------------
# Individual pipeline stages can be turned on/off here.  Note that a later
# stage won't be able to run unless data products have already been
# produced from the prior stage.

# Science processing
dodet1 = True  # calwebb_detector1
doimage2 = True  # calwebb_image2
docoron3 = True  # calwebb_coron3
docoron3_psflib = True # Custom extension to calwebb_coron3

In [3]:
sci_dir  = "/home/cperrot/Work/raw/JWST/Target/MAST/P04050/"
data_dir = "/home/cperrot/Work/raw/JWST/Target/MAST/P04050/"

# Define output subdirectories to keep science data products organized

uncal_dir  = os.path.join(sci_dir, 'uncal')  # Uncalibrated pipeline inputs should be here
det1_dir   = os.path.join(sci_dir, 'stage1')  # calwebb_detector1 pipeline outputs will go here
image2_dir = os.path.join(sci_dir, 'stage2')  # calwebb_image2 pipeline outputs will go here


# We need to check that the desired output directories exist, and if not create them
if not os.path.exists(det1_dir):
    os.makedirs(det1_dir)
if not os.path.exists(image2_dir):
    os.makedirs(image2_dir)


In [4]:
# List uncal files
uncal_files   = sorted(glob.glob(os.path.join(uncal_dir, 'obs*/*_uncal.fits')))
uncal_TAfiles = sorted(glob.glob(os.path.join(uncal_dir, 'obs*/TA/*_uncal.fits')))
print(len(uncal_files),"files")
print(len(uncal_TAfiles),"TA files")



492 files
35 TA files


In [5]:
#### STAGE 1

In [ ]:
# Set up a dictionary to define how the Detector1 pipeline should be configured

# Boilerplate dictionary setup
det1dict = {}
det1dict['group_scale'], det1dict['dq_init'], det1dict['saturation'] = {}, {}, {}
det1dict['ipc'], det1dict['superbias'], det1dict['refpix'] = {}, {}, {}
det1dict['linearity'], det1dict['persistence'], det1dict['dark_current'], = {}, {}, {}
det1dict['charge_migration'], det1dict['jump'], det1dict['clean_flicker_noise'] = {}, {}, {}
det1dict['ramp_fit'], det1dict['gain_scale'] = {}, {}

# Overrides for whether or not certain steps should be skipped
# skipping the persistence step
det1dict['persistence']['skip'] = True

# Overrides for various reference files
# Files should be in the base local directory or provide full path
#det1dict['dq_init']['override_mask'] = 'myfile.fits'  # Bad pixel mask
#det1dict['saturation']['override_saturation'] = 'myfile.fits'  # Saturation
#det1dict['reset']['override_reset'] = 'myfile.fits'  # Reset
#det1dict['linearity']['override_linearity'] = 'myfile.fits'  # Linearity
#det1dict['rscd']['override_rscd'] = 'myfile.fits'  # RSCD
#det1dict['dark_current']['override_dark'] = 'myfile.fits'  # Dark current subtraction
#det1dict['jump']['override_gain'] = 'myfile.fits'  # Gain used by jump step
#det1dict['ramp_fit']['override_gain'] = 'myfile.fits'  # Gain used by ramp fitting step
#det1dict['jump']['override_readnoise'] = 'myfile.fits'  # Read noise used by jump step
#det1dict['ramp_fit']['override_readnoise'] = 'myfile.fits'  # Read noise used by ramp fitting step

# Turn on multi-core processing (This is off by default). Choose what fraction
# of cores to use (quarter, half, all, or an integer number)
det1dict['jump']['maximum_cores'] = 'half'

# Explicitly turn on snowball correction. (Even though it is on by default)
det1dict['jump']['expand_large_events'] = True

# Turn on 1/f correction if desired
# For guidance see https://jwst-docs.stsci.edu/known-issues-with-jwst-data/1-f-noise
#det1dict['clean_flicker_noise']['skip'] = False
#det1dict['clean_flicker_noise']['fit_method'] = 'median' # 'median' or 'fft'
#det1dict['clean_flicker_noise']['background_method'] = 'median' # 'median' or 'model'
#det1dict['clean_flicker_noise']['fit_by_channel'] = False

In [11]:


# Run Detector1 stage of pipeline, specifying:
# output directory to save *_rate.fits files
# save_results flag set to True so the rate files are saved
if dodet1:
    for uncal in uncal_files:
        rate_result = Detector1Pipeline.call(uncal, output_dir=det1_dir, steps=det1dict, save_results=True)
else:
    print('Skipping Detector1 processing')

# for TA files
if dodet1:
    for uncal in uncal_TAfiles:
        rate_result = Detector1Pipeline.call(uncal, output_dir=det1_dir, steps=det1dict, save_results=True)
else:
    print('Skipping Detector1 processing')



2026-07-22 17:17:42,702 - CRDS - ERROR -  Error determining best reference for 'pars-darkcurrentstep'  =   No match found.
2026-07-22 17:17:42,706 - stpipe.step - INFO - PARS-JUMPSTEP parameters found: /home/cperrot/crds_cache/references/jwst/nircam/jwst_nircam_pars-jumpstep_0003.asdf


2026-07-22 17:17:42,721 - stpipe.pipeline - INFO - PARS-DETECTOR1PIPELINE parameters found: /home/cperrot/crds_cache/references/jwst/nircam/jwst_nircam_pars-detector1pipeline_0008.asdf
2026-07-22 17:17:42,738 - stpipe.step - INFO - Detector1Pipeline instance created.
2026-07-22 17:17:42,742 - stpipe.step - INFO - GroupScaleStep instance created.
2026-07-22 17:17:42,743 - stpipe.step - INFO - DQInitStep instance created.
2026-07-22 17:17:42,745 - stpipe.step - INFO - EmiCorrStep instance created.
2026-07-22 17:17:42,749 - stpipe.step - INFO - SaturationStep instance created.
2026-07-22 17:17:42,753 - stpipe.step - INFO - IPCStep instance created.
2026-07-22 17:17:42,755 - stpipe.step - INFO - SuperBiasStep instance created.
2026-07-22 17:17:42,756 - stpipe.step - INFO - RefPixStep instance created.
2026-07-22 17:17:42,758 - stpipe.step - INFO - RscdStep instance created.
2026-07-22 17:17:42,760 - stpipe.step - INFO - FirstFrameStep instance created.
2026-07-22 17:17:42,761 - stpipe.step

KeyboardInterrupt: 

In [22]:
# STAGE 2

In [10]:
# Set up a dictionary to define how the Image2 pipeline should be configured.

# Boilerplate dictionary setup
image2dict = {}
image2dict['assign_wcs'], image2dict['flat_field'] = {}, {}
image2dict['photom'], image2dict['resample'] = {}, {}

# Overrides for whether or not certain steps should be skipped (example)
#image2dict['resample']['skip'] = False
#image2dict['flat_field']['skip'] = True

# Overrides for various reference files
# Files should be in the base local directory or provide full path
#image2dict['assign_wcs']['override_distortion'] = 'myfile.asdf'  # Spatial distortion (ASDF file)
#image2dict['assign_wcs']['override_filteroffset'] = 'myfile.asdf'  # Imager filter offsets (ASDF file)
#image2dict['assign_wcs']['override_specwcs'] = 'myfile.asdf'  # Spectral distortion (ASDF file)
#image2dict['assign_wcs']['override_wavelengthrange'] = 'myfile.asdf'  # Wavelength channel mapping (ASDF file)
#image2dict['flat_field']['override_flat'] = 'myfile.fits'  # Pixel flatfield
#image2dict['photom']['override_photom'] = 'myfile.fits'  # Photometric calibration array

In [11]:
sstring = os.path.join(det1_dir, 'jw*rate.fits')  # Use files from the detector1 output folder
rate_files = sorted(glob.glob(sstring))
rate_files = [os.path.abspath(fname) for fname in rate_files]

print(f"Found  {len(rate_files)} science files")


Found  50 science files


In [12]:
# Run Image2 stage of pipeline, specifying:
# output directory to save *_calints.fits files
# save_results flag set to True so the calints files are saved

if doimage2:
    for rate in rate_files:
        cal_result = Image2Pipeline.call(rate, output_dir=image2_dir, steps=image2dict, save_results=True)
else:
    print("Skipping Image2 processing.")



2026-07-22 15:58:27,405 - stpipe.step - INFO - PARS-RESAMPLESTEP parameters found: /home/cperrot/crds_cache/references/jwst/nircam/jwst_nircam_pars-resamplestep_0001.asdf
2026-07-22 15:58:27,418 - stpipe.pipeline - INFO - PARS-IMAGE2PIPELINE parameters found: /home/cperrot/crds_cache/references/jwst/nircam/jwst_nircam_pars-image2pipeline_0003.asdf
2026-07-22 15:58:27,433 - stpipe.step - INFO - Image2Pipeline instance created.
2026-07-22 15:58:27,438 - stpipe.step - INFO - BackgroundStep instance created.
2026-07-22 15:58:27,443 - stpipe.step - INFO - AssignWcsStep instance created.
2026-07-22 15:58:27,445 - stpipe.step - INFO - FlatFieldStep instance created.
2026-07-22 15:58:27,446 - stpipe.step - INFO - PhotomStep instance created.
2026-07-22 15:58:27,450 - stpipe.step - INFO - ResampleStep instance created.
2026-07-22 15:58:27,584 - stpipe.step - INFO - Step Image2Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/MAST/P01411/stage1/jw01411021001_02102_00001_nrca2_rate.